In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from scipy import sparse
from lifelines import CoxPHFitter
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    get_ipython
except NameError:
    plt.switch_backend('Agg')

sns.set_theme(style='whitegrid')


In [ ]:
# Locate project paths whether the notebook is run from the project root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path(r'D:\Rishit\Projects\Cancer-Analysis')

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'seer' / 'seer_data_full.txt'
TCGA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'tcga'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

with DATA_PATH.open('r', newline='', encoding='utf-8') as f:
    seer_header = next(csv.reader(f))

def find_receptor_columns(columns):
    found = {}
    for col in columns:
        clean = re.sub(r'[^a-z0-9]+', ' ', col.lower()).strip()
        if 'her2' in clean or 'her 2' in clean:
            found.setdefault('her2_status', col)
        elif 'estrogen' in clean or re.search(r'\ber\b', clean):
            found.setdefault('er_status', col)
        elif 'progesterone' in clean or re.search(r'\bpr\b', clean):
            found.setdefault('pr_status', col)
    return found

found_receptor_columns = find_receptor_columns(seer_header)
if found_receptor_columns:
    print('Found SEER receptor-status columns:')
    for feature_name, source_name in found_receptor_columns.items():
        print(f'  {feature_name}: {source_name}')
else:
    print('No ER, PR, or HER2 receptor-status columns found in seer_data_full.txt header.')

base_raw_columns = [
    'Age recode with <1 year olds and 90+',
    'Sex',
    'Year of diagnosis',
    'Race recode (W, B, AI, API)',
    'Primary Site - labeled',
    'Histologic Type ICD-O-3',
    'Grade Recode (thru 2017)',
    'Summary stage 2000 (1998-2017)',
    'Radiation recode',
    'Chemotherapy recode (yes, no/unk)',
    'Survival months',
    'Vital status recode (study cutoff used)',
    'Median household income inflation adj to 2023',
    'Rural-Urban Continuum Code',
    'LN Positive Axillary Level I-II Recode (2010+)',
    'Response to Neoadjuvant Therapy Recode (2010+)',
]
raw_columns = base_raw_columns + list(found_receptor_columns.values())

rename_columns = {
    'Age recode with <1 year olds and 90+': 'age',
    'Sex': 'sex',
    'Year of diagnosis': 'diagnosis_year',
    'Race recode (W, B, AI, API)': 'race',
    'Primary Site - labeled': 'primary_site',
    'Histologic Type ICD-O-3': 'histologic_type',
    'Grade Recode (thru 2017)': 'grade',
    'Summary stage 2000 (1998-2017)': 'stage',
    'Radiation recode': 'radiation',
    'Chemotherapy recode (yes, no/unk)': 'chemotherapy',
    'Survival months': 'survival_months',
    'Vital status recode (study cutoff used)': 'vital_status',
    'Median household income inflation adj to 2023': 'income',
    'Rural-Urban Continuum Code': 'rural_urban',
    'LN Positive Axillary Level I-II Recode (2010+)': 'axillary_nodes_positive',
    'Response to Neoadjuvant Therapy Recode (2010+)': 'neoadjuvant_response',
}
rename_columns.update({source: feature for feature, source in found_receptor_columns.items()})
receptor_feature_names = list(found_receptor_columns.keys())

breast_chunks = []
for chunk in pd.read_csv(DATA_PATH, usecols=raw_columns, dtype=str, chunksize=250_000, low_memory=False):
    mask = chunk['Primary Site - labeled'].str.contains('Breast', case=False, na=False)
    if mask.any():
        breast_chunks.append(chunk.loc[mask].rename(columns=rename_columns))

df_breast = pd.concat(breast_chunks, ignore_index=True)

print(f'Breast cancer records loaded: {df_breast.shape[0]:,}')
print(f'Columns used from expanded SEER export: {len(raw_columns)}')
print(df_breast.head())


In [ ]:
# Profile the selected expanded fields in the breast cancer subset.
blank_tokens = {'', 'Blank(s)', 'Unknown', 'Unknown/unstaged', 'No/Unknown', 'Not Available', 'Not applicable'}
profile_rows = []
for col in df_breast.columns:
    s = df_breast[col].fillna('').astype(str).str.strip()
    nonblank = ~s.isin(blank_tokens)
    top_values = s[nonblank].value_counts().head(5)
    profile_rows.append({
        'column': col,
        'nonblank_count': int(nonblank.sum()),
        'nonblank_pct': float(nonblank.mean() * 100),
        'top_values': '; '.join([f'{idx}: {val}' for idx, val in top_values.items()]),
    })

column_profile = pd.DataFrame(profile_rows).sort_values('nonblank_pct', ascending=False)
pd.set_option('display.max_colwidth', 160)
display(column_profile)


**Feature selection note**

The expanded export is mostly broad SEER site-specific fields. For breast cancer survival prediction, the useful added fields are treatment (`radiation`, `chemotherapy`, neoadjuvant response), socioeconomic context (`income`, `rural_urban`), and tumor biology / disease extent (`histologic_type`, `grade`, axillary-node positivity). Other site-specific fields were blank or unrelated for almost all breast records and are intentionally excluded.

The notebook also searches the SEER header for ER, PR, and HER2 receptor-status columns. If those columns are present in a future export they are automatically added as categorical model features. In the current `seer_data_full.txt`, no dedicated ER, PR, or HER2 receptor-status columns were found.

The model keeps only diagnoses through 2017. Later diagnoses can include patients who died before 60 months but cannot yet include comparable observed 5-year survivors, which would bias a 5-year survival target.


In [ ]:
def parse_income_midpoint(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('$', '').replace(',', '').strip()
    if not text or text == 'Blank(s)' or not text[0].isdigit():
        return np.nan
    if '+' in text:
        return float(text.replace('+', ''))
    if '-' in text:
        low, high = text.split('-', 1)
        return (float(low.strip()) + float(high.strip())) / 2
    return np.nan

rurality_map = {
    'Counties in metropolitan areas ge 1 million pop': 1,
    'Counties in metropolitan areas of 250,000 to 1 million pop': 2,
    'Counties in metropolitan areas of lt 250 thousand pop': 3,
    'Nonmetropolitan counties adjacent to a metropolitan area': 4,
    'Nonmetropolitan counties not adjacent to a metropolitan area': 5,
}

def clean_grade(value):
    if pd.isna(value) or value in {'Blank(s)', 'Unknown'}:
        return 'Unknown/other'
    text = str(value)
    if 'Well differentiated' in text:
        return 'Well differentiated / Grade I'
    if 'Moderately differentiated' in text:
        return 'Moderately differentiated / Grade II'
    if 'Poorly differentiated' in text:
        return 'Poorly differentiated / Grade III'
    if 'Undifferentiated' in text or 'anaplastic' in text:
        return 'Undifferentiated / Grade IV'
    return 'Other/hematologic grade code'

def clean_category(value):
    if pd.isna(value):
        return 'Unknown/not documented'
    text = str(value).strip()
    if text in {'', 'Blank(s)', 'Unknown'}:
        return 'Unknown/not documented'
    return text

# Convert core fields and create a 5-year survival target where the 5-year outcome is observable.
df_breast['diagnosis_year'] = pd.to_numeric(df_breast['diagnosis_year'], errors='coerce')
df_breast['survival_months'] = pd.to_numeric(df_breast['survival_months'], errors='coerce')
df_breast['age_clean'] = df_breast['age'].str.extract(r'(\d+)').astype(float)
df_breast['income_midpoint'] = df_breast['income'].apply(parse_income_midpoint)
df_breast['rurality_score'] = df_breast['rural_urban'].map(rurality_map)
df_breast['grade_clean'] = df_breast['grade'].apply(clean_grade)

for col in ['race', 'sex', 'primary_site', 'histologic_type', 'radiation', 'chemotherapy',
            'axillary_nodes_positive', 'neoadjuvant_response'] + receptor_feature_names:
    df_breast[col] = df_breast[col].apply(clean_category)

df_breast['known_5yr_outcome'] = (
    (df_breast['survival_months'] >= 60) |
    ((df_breast['vital_status'] == 'Dead') & (df_breast['survival_months'] < 60))
)
df_breast['survived_5yr'] = (df_breast['survival_months'] >= 60).astype(int)

# Stage is modeled as ordered disease extent, so only valid ordered stages are retained.
df_ml = df_breast[
    (df_breast['diagnosis_year'] <= 2017) &
    (df_breast['race'] != 'Unknown/not documented') &
    (df_breast['stage'].isin(['Localized', 'Regional', 'Distant'])) &
    (df_breast['survival_months'].notna()) &
    (df_breast['known_5yr_outcome'])
].copy()

print(f'Patients for ML: {df_ml.shape[0]:,}')
print('\n5-year survival breakdown:')
print(df_ml['survived_5yr'].value_counts().rename(index={0: 'Died before 5 years', 1: 'Survived at least 5 years'}))
print('\nRace distribution:')
print(df_ml['race'].value_counts())


**Cox proportional hazards model**

This survival model estimates adjusted hazard ratios for mortality using age, stage, race, sex, radiation, chemotherapy, grade, income, and rurality. White patients are the reference group for race, so the Black race coefficient is interpreted as the mortality hazard for Black patients relative to otherwise similar White patients in this SEER breast cancer cohort.

In [ ]:
# Cox Proportional Hazards regression using SEER breast cancer survival time and censoring.
cox_covariates = [
    'age_clean',
    'stage',
    'race',
    'sex',
    'radiation',
    'chemotherapy',
    'grade_clean',
    'income_midpoint',
    'rurality_score',
]

cox_df = df_breast[
    (df_breast['diagnosis_year'] <= 2017) &
    (df_breast['stage'].isin(['Localized', 'Regional', 'Distant'])) &
    (df_breast['race'].isin(['White', 'Black', 'Asian or Pacific Islander', 'American Indian/Alaska Native'])) &
    (df_breast['survival_months'].notna()) &
    (df_breast['survival_months'] >= 0)
].copy()

# SEER records deaths in the month of diagnosis as 0 months; use a small positive duration for Cox fitting.
cox_df['duration_months'] = df_breast.loc[cox_df.index, 'survival_months'].clip(lower=0.5)
cox_df['event'] = (cox_df['vital_status'] == 'Dead').astype(int)
cox_df['race'] = pd.Categorical(
    cox_df['race'],
    categories=['White', 'Black', 'Asian or Pacific Islander', 'American Indian/Alaska Native'],
)
cox_df['stage'] = pd.Categorical(cox_df['stage'], categories=['Localized', 'Regional', 'Distant'])
cox_df['sex'] = pd.Categorical(cox_df['sex'], categories=['Female', 'Male'])
cox_df['chemotherapy'] = pd.Categorical(cox_df['chemotherapy'], categories=['No/Unknown', 'Yes'])
cox_df['grade_clean'] = pd.Categorical(
    cox_df['grade_clean'],
    categories=[
        'Well differentiated / Grade I',
        'Moderately differentiated / Grade II',
        'Poorly differentiated / Grade III',
        'Undifferentiated / Grade IV',
        'Other/hematologic grade code',
        'Unknown/other',
    ],
)

cox_model_df = cox_df[['duration_months', 'event'] + cox_covariates].dropna().copy()
cox_design = pd.get_dummies(
    cox_model_df,
    columns=['stage', 'race', 'sex', 'radiation', 'chemotherapy', 'grade_clean'],
    drop_first=True,
    dtype=float,
)
cox_design.columns = [re.sub(r'[^0-9A-Za-z_]+', '_', col).strip('_') for col in cox_design.columns]

cox_model = CoxPHFitter(penalizer=0.01)
cox_model.fit(cox_design, duration_col='duration_months', event_col='event')

cox_summary = cox_model.summary.reset_index().rename(columns={
    'covariate': 'term',
    'exp(coef)': 'hazard_ratio',
    'exp(coef) lower 95%': 'ci_lower_95',
    'exp(coef) upper 95%': 'ci_upper_95',
    'p': 'p_value',
})
cox_summary['is_race_term'] = cox_summary['term'].str.startswith('race_')
cox_summary_display = cox_summary[
    ['term', 'hazard_ratio', 'ci_lower_95', 'ci_upper_95', 'p_value', 'is_race_term']
].sort_values(['is_race_term', 'term'], ascending=[False, True])

print(f'Cox PH cohort: {cox_model_df.shape[0]:,} patients; deaths/events: {cox_model_df["event"].sum():,}')
print('Adjusted hazard ratios. Race rows are highlighted; White is the reference group.')
display(cox_summary_display.style.apply(
    lambda row: ['background-color: #fff2cc; font-weight: 600' if row['is_race_term'] else '' for _ in row],
    axis=1,
).format({
    'hazard_ratio': '{:.3f}',
    'ci_lower_95': '{:.3f}',
    'ci_upper_95': '{:.3f}',
    'p_value': '{:.3g}',
}))

plot_df = cox_summary_display.copy()
plot_df['label'] = plot_df['term'].str.replace('_', ' ', regex=False)
plot_df = plot_df.sort_values('hazard_ratio')
colors = np.where(plot_df['is_race_term'], '#D55E00', '#0072B2')

fig_height = max(7, 0.34 * len(plot_df))
fig, ax = plt.subplots(figsize=(10, fig_height))
y_pos = np.arange(len(plot_df))
ax.errorbar(
    plot_df['hazard_ratio'],
    y_pos,
    xerr=[
        plot_df['hazard_ratio'] - plot_df['ci_lower_95'],
        plot_df['ci_upper_95'] - plot_df['hazard_ratio'],
    ],
    fmt='none',
    ecolor='#666666',
    elinewidth=1,
    capsize=3,
    zorder=1,
)
ax.scatter(plot_df['hazard_ratio'], y_pos, color=colors, s=42, zorder=2)
ax.axvline(1, color='#222222', linestyle='--', linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df['label'])
ax.set_xlabel('Adjusted hazard ratio for death (log scale)')
ax.set_title('Cox Proportional Hazards Model: Adjusted Mortality Hazard Ratios', fontweight='bold')
ax.set_xscale('log')
ax.grid(axis='x', linestyle='--', alpha=0.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cox_hazard_ratios.png', dpi=150, bbox_inches='tight')
plt.close()

black_vs_white = cox_summary.loc[cox_summary['term'] == 'race_Black'].iloc[0]
print(f"Saved Cox forest plot: {FIGURES_DIR / 'cox_hazard_ratios.png'}")
print(
    'Interpretation: The adjusted hazard ratio for Black vs White patients is '
    f"{black_vs_white['hazard_ratio']:.2f} "
    f"(95% CI {black_vs_white['ci_lower_95']:.2f}-{black_vs_white['ci_upper_95']:.2f}, "
    f"p={black_vs_white['p_value']:.3g}). "
    'Because White is the reference race category, a hazard ratio above 1 means Black patients have a higher instantaneous risk of death at a given survival month after adjusting for age, stage, sex, radiation, chemotherapy, grade, income, and rurality; a value below 1 would mean a lower adjusted instantaneous risk.'
)



In [ ]:
features = [
    'age_clean',
    'diagnosis_year',
    'income_midpoint',
    'rurality_score',
    'race',
    'sex',
    'stage',
    'primary_site',
    'histologic_type',
    'grade_clean',
    'radiation',
    'chemotherapy',
    'axillary_nodes_positive',
    'neoadjuvant_response',
] + receptor_feature_names

target = 'survived_5yr'
X = df_ml[features].copy()
y = df_ml[target].copy()

numeric_features = ['age_clean', 'diagnosis_year', 'income_midpoint', 'rurality_score']
stage_feature = ['stage']
categorical_features = [
    'race',
    'sex',
    'primary_site',
    'histologic_type',
    'grade_clean',
    'radiation',
    'chemotherapy',
    'axillary_nodes_positive',
    'neoadjuvant_response',
] + receptor_feature_names

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

stage_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[['Localized', 'Regional', 'Distant']]))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown/not documented')),
    ('onehot', OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=500,
        sparse_output=True,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('stage', stage_transformer, stage_feature),
        ('categorical', categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=80,
            max_depth=18,
            min_samples_leaf=50,
            random_state=42,
            n_jobs=1,
            class_weight='balanced_subsample',
        )),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]:,} patients')
print(f'Test set: {X_test.shape[0]:,} patients')
print(f'Receptor features added: {receptor_feature_names if receptor_feature_names else "none found in SEER export"}')


In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
accuracy = model.score(X_test, y_test)
roc_auc = roc_auc_score(y_test, y_prob)

print('Model Performance:')
print(classification_report(y_test, y_pred))
print(f'Accuracy: {accuracy:.3f}')
print(f'ROC-AUC Score: {roc_auc:.3f}')


In [ ]:
def display_name(feature_name):
    name = feature_name
    replacements = {
        'numeric__age_clean': 'Age',
        'numeric__diagnosis_year': 'Year of diagnosis',
        'numeric__income_midpoint': 'Median household income',
        'numeric__rurality_score': 'Rurality score',
        'stage__stage': 'Summary stage',
        'categorical__race_': 'Race: ',
        'categorical__sex_': 'Sex: ',
        'categorical__primary_site_': 'Primary site: ',
        'categorical__histologic_type_': 'Histologic type: ',
        'categorical__grade_clean_': 'Grade: ',
        'categorical__radiation_': 'Radiation: ',
        'categorical__chemotherapy_': 'Chemotherapy: ',
        'categorical__axillary_nodes_positive_': 'Axillary nodes: ',
        'categorical__neoadjuvant_response_': 'Neoadjuvant response: ',
        'categorical__er_status_': 'ER status: ',
        'categorical__pr_status_': 'PR status: ',
        'categorical__her2_status_': 'HER2 status: ',
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    return name.replace('_', ' ')

def feature_group(feature_name):
    if feature_name.startswith('numeric__age_clean'):
        return 'Age'
    if feature_name.startswith('numeric__diagnosis_year'):
        return 'Year of diagnosis'
    if feature_name.startswith('numeric__income_midpoint'):
        return 'Median household income'
    if feature_name.startswith('numeric__rurality_score'):
        return 'Rurality'
    if feature_name.startswith('stage__stage'):
        return 'Stage'
    group_prefixes = {
        'categorical__race_': 'Race',
        'categorical__sex_': 'Sex',
        'categorical__primary_site_': 'Primary site',
        'categorical__histologic_type_': 'Histologic type',
        'categorical__grade_clean_': 'Grade',
        'categorical__radiation_': 'Radiation',
        'categorical__chemotherapy_': 'Chemotherapy',
        'categorical__axillary_nodes_positive_': 'Axillary nodes positive',
        'categorical__neoadjuvant_response_': 'Neoadjuvant response',
        'categorical__er_status_': 'ER status',
        'categorical__pr_status_': 'PR status',
        'categorical__her2_status_': 'HER2 status',
    }
    for prefix, group in group_prefixes.items():
        if feature_name.startswith(prefix):
            return group
    return 'Other'

classifier = model.named_steps['classifier']
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
raw_importances = pd.Series(classifier.feature_importances_, index=feature_names)

group_importances = (
    raw_importances.groupby([feature_group(name) for name in raw_importances.index])
    .sum()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#D55E00' if i == 'Race' else '#0072B2' for i in group_importances.index]
group_importances.plot(kind='barh', ax=ax, color=colors)
ax.set_title(f'Grouped Feature Importance for 5-Year Breast Cancer Survival\nRandom Forest Model (ROC-AUC = {roc_auc:.3f})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Random forest importance score', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()

print('Top grouped random-forest importances:')
display(group_importances.sort_values(ascending=False).to_frame('importance'))
print(f"Saved grouped feature importance plot: {FIGURES_DIR / 'feature_importance.png'}")


In [ ]:
# Explain a stratified sample of the corrected Random Forest model with SHAP values.
# SHAP on the full test set is unnecessarily slow for this dataset, so keep a stable sample for interpretation.
shap_sample_size = min(1000, len(X_test))
X_shap = X_test.sample(n=shap_sample_size, random_state=42)
X_shap_transformed = model.named_steps['preprocessor'].transform(X_shap)
if sparse.issparse(X_shap_transformed):
    X_shap_transformed = X_shap_transformed.toarray()

feature_names = model.named_steps['preprocessor'].get_feature_names_out()
display_feature_names = [display_name(name) for name in feature_names]

explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_shap_transformed)

# For binary classification, explain the positive class: survived_5yr = 1.
if isinstance(shap_values, list):
    positive_class_shap_values = shap_values[1]
elif getattr(shap_values, 'ndim', None) == 3:
    positive_class_shap_values = shap_values[:, :, 1]
else:
    positive_class_shap_values = shap_values

plt.figure(figsize=(11, 7))
shap.summary_plot(
    positive_class_shap_values,
    X_shap_transformed,
    feature_names=display_feature_names,
    show=False,
    max_display=25,
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()

mean_abs_shap = pd.Series(
    np.abs(positive_class_shap_values).mean(axis=0),
    index=display_feature_names,
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
mean_abs_shap.tail(25).plot(kind='barh', ax=ax, color='#009E73')
ax.set_title('Top Mean Absolute SHAP Values for 5-Year Survival Predictions', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean |SHAP value|', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_bar.png', dpi=150, bbox_inches='tight')
plt.close()

print('Top SHAP features:')
display(mean_abs_shap.sort_values(ascending=False).head(20).to_frame('mean_abs_shap'))
print('Saved SHAP plots:')
print(FIGURES_DIR / 'shap_summary.png')
print(FIGURES_DIR / 'shap_bar.png')


In [ ]:
# SHAP interaction values on a small stable sample.
# Full interaction matrices are O(features^2), so this uses the same fitted model with a compact sample.
interaction_sample_size = min(120, len(X_test))
X_interaction = X_test.sample(n=interaction_sample_size, random_state=7)
X_interaction_transformed = model.named_steps['preprocessor'].transform(X_interaction)
if sparse.issparse(X_interaction_transformed):
    X_interaction_transformed = X_interaction_transformed.toarray()

try:
    shap_interactions = explainer.shap_interaction_values(X_interaction_transformed, tree_limit=40)
except TypeError:
    shap_interactions = explainer.shap_interaction_values(X_interaction_transformed)

if isinstance(shap_interactions, list):
    positive_class_interactions = shap_interactions[1]
elif getattr(shap_interactions, 'ndim', None) == 4:
    positive_class_interactions = shap_interactions[:, :, :, 1]
else:
    positive_class_interactions = shap_interactions

mean_abs_interaction = np.abs(positive_class_interactions).mean(axis=0)
np.fill_diagonal(mean_abs_interaction, 0)
interaction_df = pd.DataFrame(mean_abs_interaction, index=display_feature_names, columns=display_feature_names)

upper_pairs = []
for i in range(interaction_df.shape[0]):
    for j in range(i + 1, interaction_df.shape[1]):
        upper_pairs.append({
            'feature_1': interaction_df.index[i],
            'feature_2': interaction_df.columns[j],
            'mean_abs_interaction': interaction_df.iat[i, j],
        })
interaction_pairs = pd.DataFrame(upper_pairs).sort_values('mean_abs_interaction', ascending=False)
interaction_pairs['pair'] = interaction_pairs['feature_1'] + ' x ' + interaction_pairs['feature_2']

fig, ax = plt.subplots(figsize=(10, 7))
interaction_pairs.head(20).sort_values('mean_abs_interaction').plot(
    x='pair', y='mean_abs_interaction', kind='barh', ax=ax, color='#CC79A7', legend=False
)
ax.set_title('Top SHAP Interaction Values for 5-Year Survival Predictions', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean |SHAP interaction value|', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_interaction_bar.png', dpi=150, bbox_inches='tight')
plt.close()

top_interaction_features = pd.unique(interaction_pairs.head(10)[['feature_1', 'feature_2']].to_numpy().ravel())[:12]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    interaction_df.loc[top_interaction_features, top_interaction_features],
    cmap='viridis', ax=ax, cbar_kws={'label': 'Mean |SHAP interaction value|'}
)
ax.set_title('SHAP Interaction Heatmap: Top Interacting Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_interaction_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()

print('Top SHAP interaction pairs:')
display(interaction_pairs.head(20))
print('Saved SHAP interaction plots:')
print(FIGURES_DIR / 'shap_interaction_bar.png')
print(FIGURES_DIR / 'shap_interaction_heatmap.png')


In [ ]:
# External validation: map TCGA-BRCA clinical data onto the SEER-trained model schema.
TCGA_MISSING = {'--', "'--", '', 'not reported', 'Not Reported', 'unknown', 'Unknown', 'NaN', 'nan'}

def tcga_clean(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    return np.nan if text in TCGA_MISSING else text

def tcga_numeric(value):
    value = tcga_clean(value)
    return pd.to_numeric(value, errors='coerce')

def tcga_map_race(value):
    value = tcga_clean(value)
    if pd.isna(value):
        return 'Unknown/not documented'
    text = str(value).lower()
    if 'white' in text:
        return 'White'
    if 'black' in text or 'african' in text:
        return 'Black'
    if 'asian' in text:
        return 'Asian or Pacific Islander'
    if 'american indian' in text or 'alaska native' in text:
        return 'American Indian/Alaska Native'
    return 'Unknown/not documented'

def tcga_map_sex(value):
    value = tcga_clean(value)
    if pd.isna(value):
        return 'Unknown/not documented'
    return str(value).strip().title()

def tcga_map_stage(value):
    value = tcga_clean(value)
    if pd.isna(value):
        return np.nan
    text = str(value).upper()
    if 'IV' in text:
        return 'Distant'
    if 'III' in text or 'II' in text:
        return 'Regional'
    if ' I' in f' {text}' or text in {'STAGE I', 'STAGE IA', 'STAGE IB', 'STAGE 0', 'STAGE 0IS'}:
        return 'Localized'
    return np.nan

def tcga_map_radiation(group):
    treatment_type = group['treatments.treatment_type'].fillna('').astype(str)
    therapy = group['treatments.treatment_or_therapy'].fillna('').astype(str).str.lower()
    received = therapy.eq('yes')
    if ((treatment_type.str.contains('Radiation', case=False, na=False)) & received).any():
        return 'Beam radiation'
    if ((treatment_type.str.contains('Radiation', case=False, na=False)) & therapy.eq('no')).any():
        return 'None/Unknown'
    return 'Unknown/not documented'

def tcga_map_chemo(group):
    treatment_type = group['treatments.treatment_type'].fillna('').astype(str)
    therapy = group['treatments.treatment_or_therapy'].fillna('').astype(str).str.lower()
    received = therapy.eq('yes')
    chemo_like = treatment_type.str.contains('Chemotherapy|Pharmaceutical', case=False, na=False)
    if (chemo_like & received).any():
        return 'Yes'
    if (chemo_like & therapy.eq('no')).any():
        return 'No/Unknown'
    return 'Unknown/not documented'

def tcga_map_nodes(value):
    n = pd.to_numeric(tcga_clean(value), errors='coerce')
    if pd.isna(n):
        return 'Not documented; Not assessed or unknown if assessed'
    if n <= 0:
        return 'All ipsilateral axillary nodes examined negative'
    if n < 100:
        return f'{int(n):02d}'
    return str(int(n))

def tcga_group_first(group, col):
    vals = group[col].map(tcga_clean).dropna()
    return vals.iloc[0] if len(vals) else np.nan

clinical = pd.read_csv(TCGA_DIR / 'clinical.tsv', sep='\t', dtype=str)
pathology = pd.read_csv(TCGA_DIR / 'pathology_detail.tsv', sep='\t', dtype=str)

case_rows = []
for case_id, group in clinical.groupby('cases.submitter_id'):
    vital_status = tcga_group_first(group, 'demographic.vital_status')
    days_to_death = group['demographic.days_to_death'].map(tcga_numeric).max()
    days_to_follow = group['diagnoses.days_to_last_follow_up'].map(tcga_numeric).max()
    observed_days = days_to_death if str(vital_status).lower() == 'dead' and pd.notna(days_to_death) else days_to_follow

    age = group['demographic.age_at_index'].map(tcga_numeric).dropna()
    age_clean = age.iloc[0] if len(age) else group['diagnoses.age_at_diagnosis'].map(tcga_numeric).dropna().iloc[0] / 365.25 if len(group['diagnoses.age_at_diagnosis'].map(tcga_numeric).dropna()) else np.nan
    path_stage = tcga_group_first(group, 'diagnoses.ajcc_pathologic_stage')
    morphology = tcga_group_first(group, 'diagnoses.morphology')
    histologic_type = str(morphology).split('/')[0] if pd.notna(morphology) and str(morphology)[0].isdigit() else 'Unknown/not documented'

    case_rows.append({
        'case_id': case_id,
        'age_clean': age_clean,
        'diagnosis_year': tcga_numeric(tcga_group_first(group, 'diagnoses.year_of_diagnosis')),
        'income_midpoint': np.nan,
        'rurality_score': np.nan,
        'race': tcga_map_race(tcga_group_first(group, 'demographic.race')),
        'sex': tcga_map_sex(tcga_group_first(group, 'demographic.gender')),
        'stage': tcga_map_stage(path_stage),
        'primary_site': 'C50.9-Breast, NOS',
        'histologic_type': histologic_type,
        'grade_clean': 'Unknown/other',
        'radiation': tcga_map_radiation(group),
        'chemotherapy': tcga_map_chemo(group),
        'neoadjuvant_response': 'Unknown/not documented',
        'observed_days': observed_days,
        'vital_status': vital_status,
    })

tcga_cases = pd.DataFrame(case_rows)
node_by_case = pathology.groupby('cases.submitter_id')['pathology_details.lymph_nodes_positive'].first().map(tcga_map_nodes)
tcga_cases['axillary_nodes_positive'] = tcga_cases['case_id'].map(node_by_case).fillna('Not documented; Not assessed or unknown if assessed')
for receptor_feature in receptor_feature_names:
    tcga_cases[receptor_feature] = 'Unknown/not documented'

tcga_cases['known_5yr_outcome'] = (
    (tcga_cases['observed_days'] >= 1825) |
    ((tcga_cases['vital_status'].astype(str).str.lower() == 'dead') & (tcga_cases['observed_days'] < 1825))
)
tcga_cases['survived_5yr'] = (tcga_cases['observed_days'] >= 1825).astype(int)

tcga_validation = tcga_cases[
    tcga_cases['known_5yr_outcome'] &
    tcga_cases['stage'].isin(['Localized', 'Regional', 'Distant']) &
    (tcga_cases['race'] != 'Unknown/not documented')
].copy()
X_tcga = tcga_validation[features]
y_tcga = tcga_validation['survived_5yr']

tcga_prob = model.predict_proba(X_tcga)[:, 1]
tcga_pred = model.predict(X_tcga)
tcga_accuracy = model.score(X_tcga, y_tcga)
tcga_roc_auc = roc_auc_score(y_tcga, tcga_prob) if y_tcga.nunique() == 2 else np.nan

print(f'TCGA-BRCA cases loaded: {tcga_cases.shape[0]:,}')
print(f'TCGA external validation cases with known 5-year outcome and mapped features: {tcga_validation.shape[0]:,}')
print('\nTCGA 5-year survival breakdown:')
print(y_tcga.value_counts().rename(index={0: 'Died before 5 years', 1: 'Survived at least 5 years'}))
print('\nTCGA External Validation Performance:')
print(classification_report(y_tcga, tcga_pred))
print(f'TCGA Accuracy: {tcga_accuracy:.3f}')
print(f'TCGA ROC-AUC Score: {tcga_roc_auc:.3f}')

fpr, tpr, _ = roc_curve(y_tcga, tcga_prob)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#0072B2', lw=2, label=f'TCGA ROC-AUC = {tcga_roc_auc:.3f}')
ax.plot([0, 1], [0, 1], color='0.5', linestyle='--', lw=1)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('External Validation ROC: SEER-Trained Model on TCGA-BRCA', fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'tcga_external_validation_roc.png', dpi=150, bbox_inches='tight')
plt.close()

calibration_df = pd.DataFrame({'probability': tcga_prob, 'actual': y_tcga})
calibration_df['risk_decile'] = pd.qcut(calibration_df['probability'], q=min(10, calibration_df['probability'].nunique()), duplicates='drop')
calibration_summary = calibration_df.groupby('risk_decile', observed=True).agg(
    mean_predicted_survival=('probability', 'mean'),
    observed_survival=('actual', 'mean'),
    cases=('actual', 'size'),
).reset_index()

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], color='0.5', linestyle='--', lw=1)
ax.scatter(calibration_summary['mean_predicted_survival'], calibration_summary['observed_survival'], s=55, color='#D55E00')
ax.plot(calibration_summary['mean_predicted_survival'], calibration_summary['observed_survival'], color='#D55E00', lw=1)
ax.set_xlabel('Mean predicted 5-year survival probability')
ax.set_ylabel('Observed 5-year survival fraction')
ax.set_title('TCGA External Validation Calibration by Prediction Decile', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'tcga_external_validation_calibration.png', dpi=150, bbox_inches='tight')
plt.close()

print('Saved TCGA validation plots:')
print(FIGURES_DIR / 'tcga_external_validation_roc.png')
print(FIGURES_DIR / 'tcga_external_validation_calibration.png')
print('\nTCGA calibration by prediction decile:')
display(calibration_summary)
